In [1]:
# 06_visualization.ipynb
# Cross-Domain AI Governance Gap Analysis
# 06. Visualization
#
# Publication figures for the noise-corrected GGI results and the sensitivity analysis.
# seaborn, grayscale palette, no titles or captions (captions live in the manuscript),
# every figure saved as both PNG and PDF at 600 dpi into results/figures.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns

RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style="white", context="paper")
plt.rcParams.update({
    "font.family": "serif", "font.size": 11,
    "axes.edgecolor": "black", "axes.linewidth": 0.8, "axes.grid": False,
    "savefig.bbox": "tight", "figure.dpi": 150,
})
DPI = 600
REG_ORDER = ["KR_AI_Basic_Act", "EU_AI_Act", "NAIC"]
REG_LABELS = {"KR_AI_Basic_Act": "KR AI Basic Act", "EU_AI_Act": "EU AI Act", "NAIC": "NAIC"}
DATASET_ORDER = ["German Credit", "Credit Card Approval", "Medical Cost", "Porto Seguro"]

# Save a figure as PNG and PDF at 600 dpi; no title is ever added (captions are external).
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=DPI)
    plt.close(fig)

ggi_full = pd.read_csv(os.path.join(TAB_DIR, "04_ggi_full.csv"))                 # corrected (primary)
ggi_orig = pd.read_csv(os.path.join(TAB_DIR, "04_ggi_full_original.csv"))         # original
cmp_df = pd.read_csv(os.path.join(TAB_DIR, "04_ggi_corrected_vs_original.csv"))
porto = pd.read_csv(os.path.join(TAB_DIR, "04_porto_proxy_significance.csv"))
boot = pd.read_csv(os.path.join(TAB_DIR, "05_sensitivity_bootstrap.csv"))
sens_expl = pd.read_csv(os.path.join(TAB_DIR, "05_sensitivity_explainability.csv"))

# Figure 1: noise-corrected GGI heatmap (dataset x regulation), grayscale
mat = (ggi_full.pivot(index="dataset", columns="regulation", values="GGI")
       .reindex(index=DATASET_ORDER, columns=REG_ORDER))
fig, ax = plt.subplots(figsize=(5.2, 4.0))
sns.heatmap(mat, ax=ax, cmap="Greys", annot=True, fmt=".3f", linewidths=0.6,
            linecolor="black", cbar_kws={"label": "GGI"}, annot_kws={"color": "black"})
for text, val in zip(ax.texts, mat.values.flatten()):
    text.set_color("white" if val > mat.values.max() * 0.6 else "black")
ax.set_xticklabels([REG_LABELS[c] for c in REG_ORDER], rotation=20, ha="right")
ax.set_ylabel(""); ax.set_xlabel("")
save_fig(fig, "fig1_ggi_heatmap")

# Figure 2: component-gap decomposition per dataset (stacked bars, hatches), KR baseline
kr = ggi_full[ggi_full["regulation"] == "KR_AI_Basic_Act"].set_index("dataset").reindex(DATASET_ORDER)
comp_cols = ["g_PERF", "g_DI", "g_SP", "g_EXPL"]
comp_labels = ["Performance", "Disparate Impact", "Statistical Parity", "Explainability"]
grays = ["0.15", "0.40", "0.65", "0.85"]
hatches = ["", "//", "..", "xx"]
fig, ax = plt.subplots(figsize=(6.0, 4.0))
bottom = np.zeros(len(DATASET_ORDER)); x = np.arange(len(DATASET_ORDER))
for col, lab, g, h in zip(comp_cols, comp_labels, grays, hatches):
    vals = kr[col].fillna(0).values
    ax.bar(x, vals, bottom=bottom, width=0.6, label=lab, color=g,
           edgecolor="black", linewidth=0.7, hatch=h)
    bottom += vals
ax.set_xticks(x); ax.set_xticklabels(DATASET_ORDER, rotation=20, ha="right")
ax.set_ylabel("Component gap contribution")
ax.legend(frameon=False, fontsize=9, loc="upper left")
sns.despine(ax=ax)
save_fig(fig, "fig2_component_stack")

# Figure 3: mean GGI by domain x regulation (grouped bars), grayscale
dom = (ggi_full.groupby(["domain", "regulation"])["GGI"].mean().reset_index()
       .pivot(index="domain", columns="regulation", values="GGI")[REG_ORDER]).reindex(["credit", "insurance"])
fig, ax = plt.subplots(figsize=(6.0, 4.0))
x = np.arange(len(dom.index)); width = 0.25
reg_grays = {"KR_AI_Basic_Act": "0.25", "EU_AI_Act": "0.55", "NAIC": "0.80"}
for i, reg in enumerate(REG_ORDER):
    ax.bar(x + (i - 1) * width, dom[reg].values, width, label=REG_LABELS[reg],
           color=reg_grays[reg], edgecolor="black", linewidth=0.7)
ax.set_xticks(x); ax.set_xticklabels([d.capitalize() for d in dom.index])
ax.set_ylabel("Mean GGI"); ax.legend(frameon=False, fontsize=9)
sns.despine(ax=ax)
save_fig(fig, "fig3_domain_bars")

# Figure 4: noise-corrected GGI with bootstrap 95% CI per dataset (KR baseline), grayscale
b = boot[boot["regulation"] == "KR_AI_Basic_Act"].set_index("dataset").reindex(DATASET_ORDER)
fig, ax = plt.subplots(figsize=(6.0, 4.0))
x = np.arange(len(DATASET_ORDER))
for xi, ds in zip(x, DATASET_ORDER):
    dm = b.loc[ds, "domain"]
    ax.errorbar(xi, b.loc[ds, "GGI_mean"],
                yerr=[[b.loc[ds, "GGI_mean"] - b.loc[ds, "CI_low"]],
                      [b.loc[ds, "CI_high"] - b.loc[ds, "GGI_mean"]]],
                fmt="s" if dm == "insurance" else "o", color="black", ecolor="0.4",
                elinewidth=1.2, capsize=4, markersize=7,
                markerfacecolor="white" if dm == "insurance" else "black")
ax.set_xticks(x); ax.set_xticklabels(DATASET_ORDER, rotation=20, ha="right")
ax.set_ylabel("GGI (KR AI Basic Act)")
ax.legend(handles=[Line2D([0], [0], marker="o", color="black", markerfacecolor="black",
                          linestyle="", markersize=7, label="Credit"),
                   Line2D([0], [0], marker="s", color="black", markerfacecolor="white",
                          linestyle="", markersize=7, label="Insurance")],
          frameon=False, fontsize=9)
sns.despine(ax=ax)
save_fig(fig, "fig4_bootstrap_ci")

# Figure 5: domain-mean GGI vs explainability threshold, grayscale
fig, ax = plt.subplots(figsize=(6.0, 4.0))
ax.plot(sens_expl["tau_EXPL"], sens_expl["credit_mean"], marker="o", color="black",
        linewidth=1.5, label="Credit")
ax.plot(sens_expl["tau_EXPL"], sens_expl["insurance_mean"], marker="s", color="0.5",
        linewidth=1.5, linestyle="--", markerfacecolor="white", label="Insurance")
ax.set_xlabel(r"Explainability threshold $\tau_{\mathrm{EXPL}}$")
ax.set_ylabel("Mean GGI"); ax.legend(frameon=False, fontsize=9)
sns.despine(ax=ax)
save_fig(fig, "fig5_sensitivity_expl")

# Figure 6: original vs noise-corrected GGI by dataset (KR baseline), grayscale
go = (ggi_orig[ggi_orig["regulation"] == "KR_AI_Basic_Act"].set_index("dataset")
      .reindex(DATASET_ORDER)["GGI"].values)
gc = (ggi_full[ggi_full["regulation"] == "KR_AI_Basic_Act"].set_index("dataset")
      .reindex(DATASET_ORDER)["GGI"].values)
fig, ax = plt.subplots(figsize=(6.4, 4.0))
x = np.arange(len(DATASET_ORDER)); w = 0.38
ax.bar(x - w / 2, go, w, label="Original (all-proxy worst-case)", color="0.75",
       edgecolor="black", linewidth=0.7)
ax.bar(x + w / 2, gc, w, label="Noise-corrected (significance-gated)", color="0.30",
       edgecolor="black", linewidth=0.7)
ax.set_xticks(x); ax.set_xticklabels(DATASET_ORDER, rotation=20, ha="right")
ax.set_ylabel("GGI (KR AI Basic Act)")
ax.legend(frameon=False, fontsize=9, loc="upper left")
sns.despine(ax=ax)
save_fig(fig, "fig6_ggi_correction")

# Figure 7: Porto Seguro per-proxy disparate impact with BH significance, grayscale
ps = porto.copy().sort_values("DI")
di_disp = ps["DI"].replace(0, 0.03).values
yy = np.arange(len(ps))
sizes = np.clip(ps["n_unpriv"].values / 25.0, 10, 320)
colors = ["black" if s else "0.80" for s in ps["BH_significant"]]
fig, ax = plt.subplots(figsize=(6.6, 4.4))
ax.scatter(di_disp, yy, s=sizes, c=colors, edgecolor="black", linewidth=0.6, zorder=3)
ax.axvline(1.0, color="0.3", lw=1.0)
ax.axvline(0.8, color="0.6", lw=0.8, ls="--"); ax.axvline(1.25, color="0.6", lw=0.8, ls="--")
ax.set_xscale("log"); ax.set_xlim(0.025, 5)
ax.set_yticks(yy); ax.set_yticklabels(ps["attr"], fontsize=8)
ax.set_xlabel("Disparate-impact ratio (log scale); marker size proportional to subgroup size")
for _, r in ps.iterrows():
    if r["DI"] == 0 or r["n_unpriv"] < 100:
        ax.annotate(f"n={int(r['n_unpriv'])}",
                    (0.03 if r["DI"] == 0 else r["DI"], list(ps["attr"]).index(r["attr"])),
                    fontsize=7, color="0.25", xytext=(4, 3), textcoords="offset points")
ax.legend(handles=[Line2D([0], [0], marker="o", color="w", markerfacecolor="black",
                          markeredgecolor="black", markersize=9, label="BH-significant (FDR<0.05)"),
                   Line2D([0], [0], marker="o", color="w", markerfacecolor="0.80",
                          markeredgecolor="black", markersize=9, label="not significant")],
          frameon=False, fontsize=8, loc="upper left")
sns.despine(ax=ax)
save_fig(fig, "fig7_porto_proxy_significance")

# Figure 8: insurance/credit GGI ratio, original vs corrected, across regulations, grayscale
regs = [REG_LABELS[r] for r in cmp_df["regulation"]]
x = np.arange(len(regs)); w = 0.36
fig, ax = plt.subplots(figsize=(6.2, 4.0))
ax.bar(x - w / 2, cmp_df["ratio_original"], w, label="Original", color="0.75",
       edgecolor="black", linewidth=0.7)
ax.bar(x + w / 2, cmp_df["ratio_corrected"], w, label="Noise-corrected", color="0.30",
       edgecolor="black", linewidth=0.7)
ax.axhline(1.0, color="0.5", lw=0.8, ls=":")
ax.set_xticks(x); ax.set_xticklabels(regs, rotation=20, ha="right")
ax.set_ylabel("Insurance / Credit mean-GGI ratio")
ax.legend(frameon=False, fontsize=9)
sns.despine(ax=ax)
save_fig(fig, "fig8_domain_ratio")

print("06_visualization complete:", len([f for f in os.listdir(FIG_DIR) if f.endswith(".png")]), "PNG figures")


06_visualization complete: 8 PNG figures
